# Dental YOLO26 — full-data Colab training

This notebook downloads the two version-pinned Kaggle datasets, rebuilds the leakage-safe 38-class dataset on Colab's local SSD, applies capped class-aware augmentation for minority classes, trains YOLO26s on the full training split, and saves every epoch checkpoint plus final reports to Google Drive. The held-out test split is evaluated only after validation-based model selection.

In [ ]:
from google.colab import drive
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then reconnect.'
print(torch.cuda.get_device_name(0))
print(f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB VRAM')


In [ ]:
from pathlib import Path

repo = Path('/content/dental-yolo26-detection')
if (repo / '.git').exists():
    !git -C {repo} pull --ff-only
else:
    !git clone https://github.com/sina-04/dental-yolo26-detection.git {repo}
%cd /content/dental-yolo26-detection
!python -m pip uninstall -q -y albumentationsx
!python -m pip install -q --force-reinstall albumentations==2.0.8 albucore==0.0.24 'numpy>=2.0,<2.3'
!python -m pip install -q -r requirements-colab.txt


## Prepare and verify the datasets

KaggleHub downloads the exact dataset versions to Colab's ephemeral SSD. Public downloads often work anonymously. If Kaggle requests authentication, add a Colab secret named `KAGGLE_API_TOKEN` containing a Kaggle API token and rerun this cell. Raw data never enters the Git repository or Google Drive.

In [ ]:
!python -m src.colab_workflow \
  --stage prepare \
  --data-root /content/dental_yolo26_data \
  --augment-fraction 0.15 \
  --minority-target-instances 128 \
  --max-augmentations-per-image 8 \
  --rebuild-data \
  --seed 42


## Train on the full split

The accuracy profile uses YOLO26s, 640-pixel images, batch size 32, a 15-epoch baseline, and up to 100 class-aware medically augmented epochs with cosine-decay AdamW and 20-epoch early stopping. Outputs are written to Drive after every epoch. Rerunning the cell resumes incomplete experiments from `last.pt`. Batch size 32 is validated on a 14.6 GiB Tesla T4; reduce it to 16 or 8 if the assigned GPU runs out of memory.

In [ ]:
!python -m src.colab_workflow \
  --stage train \
  --results-root /content/drive/MyDrive/dental-yolo26-detection/colab-results \
  --model yolo26s.pt \
  --baseline-epochs 15 \
  --tuned-epochs 100 \
  --imgsz 640 \
  --batch 32 \
  --workers 4 \
  --patience 20 \
  --run-prefix colab_accuracy_v1 \
  --seed 42


In [ ]:
from pathlib import Path
results = Path('/content/drive/MyDrive/dental-yolo26-detection/colab-results')
print('Best model:', results / 'artifacts/best.pt')
print('Metrics:', results / 'reports/final_metrics.json')
print('Report:', results / 'FINAL_REPORT.md')
assert (results / 'artifacts/best.pt').exists(), 'Training has not completed yet.'
